# PCA

## Import data

In [16]:
import pandas as pd

df = pd.read_csv('../../data/base_v3.csv')
df.head()

,ISRC,Track,Album Name,Artist,Release Date,All Time Rank,Track Score,Spotify Streams,Spotify Popularity,YouTube Views,...,Pandora Streams,Pandora Track Stations,Shazam Counts,Explicit Track,Length,Releases,Genres,Registration Country,Playlist Probability,High Playlist Probability
0,QM24S2402528,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,1,725.4,390470936,92,84274754,...,18004655,22931,2669262,0,155.1510,1,[],United States,0.734589,1
1,USUG12400910,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,2,545.9,323703884,92,116347040,...,7780028,28444,1118279,1,274.1920,5,"['hip hop', 'producer tag - dj mustard']",United States,0.721077,1
2,QZJ842400387,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,3,538.4,601309283,92,122599116,...,5022621,5639,5285340,0,143.1865,8,['synth-pop'],United States,0.771418,1
3,USSM12209777,Flowers,Flowers - Single,Miley Cyrus,1/12/2023,4,444.9,2031280633,85,1096100899,...,190260277,203384,11822942,0,200.4530,30,"['yacht rock', 'alternative pop', 'pop rock', ...",United States,0.799485,1
4,USUG12403398,Houdini,Houdini,Eminem,5/31/2024,5,423.3,107034922,88,77373957,...,4493884,7006,457017,1,227.0320,20,[],United States,0.721544,1


## Remove imputed values for more representative clusters

In [17]:
basev1 = pd.read_csv('../../data/base_v1.csv')
basev1.head()

,ISRC,Track,Album Name,Artist,Release Date,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track,Length,Releases,Genres,Registration Country
0,QM24S2402528,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,1,725.4,"390,470,936","30,716","196,631,588",...,"18,004,655","22,931","4,818,457","2,669,262",NaN,0,155.1510,1,[],United States
1,USUG12400910,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,2,545.9,"323,703,884","28,113","174,597,137",...,"7,780,028","28,444","6,623,075","1,118,279",NaN,1,274.1920,5,"['hip hop', 'producer tag - dj mustard']",United States
2,QZJ842400387,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,3,538.4,"601,309,283","54,331","211,607,669",...,"5,022,621","5,639","7,208,651","5,285,340",NaN,0,143.1865,8,['synth-pop'],United States
3,USSM12209777,Flowers,Flowers - Single,Miley Cyrus,1/12/2023,4,444.9,"2,031,280,633","269,802","136,569,078",...,"190,260,277","203,384",NaN,"11,822,942",NaN,0,200.4530,30,"['yacht rock', 'alternative pop', 'pop rock', ...",United States
4,USUG12403398,Houdini,Houdini,Eminem,5/31/2024,5,423.3,"107,034,922","7,223","151,469,874",...,"4,493,884","7,006","207,179","457,017",NaN,1,227.0320,20,[],United States


In [18]:
valid_isrcs = basev1.loc[
    ~(basev1['TikTok Posts'].isnull() | 
      basev1['TikTok Likes'].isnull() | 
      basev1['TikTok Views'].isnull() |
      basev1['YouTube Views'].isnull() |
      basev1['YouTube Likes'].isnull()), 
    'ISRC'
]

df_filtered = df[df['ISRC'].isin(valid_isrcs)]
df_final = df_filtered.merge(valid_isrcs.to_frame(), on='ISRC', how='inner')
print(f'Length of dataframe with imputed values: {len(df)}')
print(f'Length of dataframe without imputed values: {len(df_final)}')

Length of dataframe with imputed values: 4572
Length of dataframe without imputed values: 3311


In [19]:
df_final['All Time Rank Bin'] = pd.qcut(df_final['All Time Rank'], q=2, labels=[1, 0]).astype(int)
df_final.head(2)

,ISRC,Track,Album Name,Artist,Release Date,All Time Rank,Track Score,Spotify Streams,Spotify Popularity,YouTube Views,...,Pandora Track Stations,Shazam Counts,Explicit Track,Length,Releases,Genres,Registration Country,Playlist Probability,High Playlist Probability,All Time Rank Bin
0,QM24S2402528,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,1,725.4,390470936,92,84274754,...,22931,2669262,0,155.151,1,[],United States,0.734589,1,1
1,USUG12400910,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,2,545.9,323703884,92,116347040,...,28444,1118279,1,274.192,5,"['hip hop', 'producer tag - dj mustard']",United States,0.721077,1,1


In [20]:
df_final['TikTok Views per Post'] = df_final['TikTok Views'] / df_final['TikTok Posts']

df_final['YouTube Engagement Rate'] = (df_final['YouTube Likes'] + df_final['YouTube Views']) / (df_final['Spotify Streams'] + 1)
df_final['TikTok Engagement Rate'] = (df_final['TikTok Likes'] + df_final['TikTok Views']) / (df_final['Spotify Streams'] + 1)

df_final['TikTok Impact'] = df_final['TikTok Views per Post'] / (df_final['YouTube Views'] + 1)

df_final['YouTube Likes per View'] = df_final['YouTube Likes'] / df_final['YouTube Views']
df_final['TikTok Likes per View'] = df_final['TikTok Likes'] / df_final['TikTok Views']
df_final['TikTok Views per Post'] = df_final['TikTok Views'] / df_final['TikTok Posts']

df_final['Release Date'] = pd.to_datetime(df_final['Release Date'], errors='coerce')
df_final['Release Date Age (days)'] = (pd.Timestamp.today() - df_final['Release Date']).dt.days

# Audience Reach Metrics
df_final['YouTube View-to-Like Ratio'] = df_final['YouTube Views'] / (df_final['YouTube Likes'] + 1)
df_final['TikTok View-to-Like Ratio'] = df_final['TikTok Views'] / (df_final['TikTok Likes'] + 1)
df_final['YouTube-TikTok View Ratio'] = df_final['YouTube Views'] / (df_final['TikTok Views'] + 1)
df_final['YouTube-TikTok Like Ratio'] = df_final['YouTube Likes'] / (df_final['TikTok Likes'] + 1)

# Virality Metrics
df_final['TikTok Engagement Multiplier'] = (df_final['TikTok Likes per View'] * df_final['TikTok Views per Post'])
df_final['YouTube Engagement Multiplier'] = (df_final['YouTube Likes per View'] * df_final['YouTube Views'])

# Content Volume Metrics
df_final['TikTok Likes per Post'] = df_final['TikTok Likes'] / (df_final['TikTok Posts'] + 1)
df_final['TikTok Engagement per Post'] = (df_final['TikTok Views'] + df_final['TikTok Likes']) / (df_final['TikTok Posts'] + 1)

# Growth Potential Metrics
df_final['YouTube Growth Score'] = df_final['YouTube Views'] / (df_final['Release Date Age (days)'] + 1)
df_final['TikTok Growth Score'] = df_final['TikTok Views'] / (df_final['Release Date Age (days)'] + 1)

# Shazam Influence
df_final['Shazam Conversion Rate'] = df_final['Shazam Counts'] / (df_final['YouTube Views'] + df_final['TikTok Views'] + 1)

df_final.head()
"""
'TikTok Views per Post', 'YouTube Engagement Rate', 'TikTok Engagement Rate', 'TikTok Impact',
'YouTube View-to-Like Ratio', 'TikTok View-to-Like Ratio', 'YouTube-TikTok View Ratio',
'YouTube-TikTok Like Ratio', 'TikTok Engagement Multiplier', 'YouTube Engagement Multiplier',
'TikTok Likes per Post', 'TikTok Engagement per Post', 'YouTube Growth Score', 'TikTok Growth Score', 'Shazam Conversion Rate'
"""

"\n'TikTok Views per Post', 'YouTube Engagement Rate', 'TikTok Engagement Rate', 'TikTok Impact',\n'YouTube View-to-Like Ratio', 'TikTok View-to-Like Ratio', 'YouTube-TikTok View Ratio',\n'YouTube-TikTok Like Ratio', 'TikTok Engagement Multiplier', 'YouTube Engagement Multiplier',\n'TikTok Likes per Post', 'TikTok Engagement per Post', 'YouTube Growth Score', 'TikTok Growth Score', 'Shazam Conversion Rate'\n"

## Perform PCA

### Import modules

In [21]:
# install plotly for interactive graphs
import subprocess
import sys
def install(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '--quiet'])
install('plotly')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import plotly.express as px
import seaborn as sns
from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler

### Apply PCA with linear and radial kernels

In [22]:
def normalize_data(df, quantitative_cols):
    """Use standard scaler to normalize quantitative data columns."""
    scaler = StandardScaler()
    return scaler.fit_transform(df[quantitative_cols])

def apply_pca(df_scaled, df, n):
    """Apply PCA to scaled data and join with categorical labels."""
    pca = PCA(n_components=n)
    df_pca = pd.DataFrame(pca.fit_transform(df_scaled), columns=[f'PC{i+1}' for i in range(n)])
    return df_pca.assign(**df[['All Time Rank', 'All Time Rank Bin', 'Track', 'Artist']])

def apply_kernel_pca(df_scaled, df, n, kernel='rbf', gamma=0.05):
    """Apply Kernel PCA to scaled data and join with categorical labels."""
    kpca = KernelPCA(n_components=n, kernel=kernel, gamma=gamma, fit_inverse_transform=True)
    df_kpca = pd.DataFrame(kpca.fit_transform(df_scaled), columns=[f'PC{i+1}' for i in range(n)])
    return df_kpca.assign(**df[['All Time Rank', 'All Time Rank Bin', 'Track', 'Artist']])

### Run above functions

In [23]:
print('Starting Data:')
display(df_final.head())

# select quantitative columns (non-streaming, social media related)
quantitative_cols = [
    'TikTok Posts', 'TikTok Views per Post', 'YouTube Engagement Rate', 'TikTok Engagement Rate', 'TikTok Impact',
    'YouTube View-to-Like Ratio', 'TikTok View-to-Like Ratio', 'YouTube-TikTok View Ratio',
    'YouTube-TikTok Like Ratio', 'TikTok Engagement Multiplier', 'YouTube Engagement Multiplier',
    'TikTok Likes per Post', 'TikTok Engagement per Post', 'YouTube Growth Score', 'TikTok Growth Score', 'Shazam Conversion Rate'
]
# quantitative_cols = [
#     'YouTube Views', 'YouTube Likes', 'TikTok Posts', 
#     'TikTok Likes', 'TikTok Views'
# ]

# normalize selected columns
df_scaled = normalize_data(df_final, quantitative_cols)

print('PCA-Ready Data:')
display(pd.DataFrame(df_scaled).head())

# fit and transform data points onto PC vector space
pca_df_2d = apply_pca(df_scaled, df_final, 2)
pca_df_3d = apply_pca(df_scaled, df_final, 3)
kpca_df_2d = apply_kernel_pca(df_scaled, df_final, 2)
kpca_df_3d = apply_kernel_pca(df_scaled, df_final, 3)

Starting Data:


,ISRC,Track,Album Name,Artist,Release Date,All Time Rank,Track Score,Spotify Streams,Spotify Popularity,YouTube Views,...,TikTok View-to-Like Ratio,YouTube-TikTok View Ratio,YouTube-TikTok Like Ratio,TikTok Engagement Multiplier,YouTube Engagement Multiplier,TikTok Likes per Post,TikTok Engagement per Post,YouTube Growth Score,TikTok Growth Score,Shazam Conversion Rate
0,QM24S2402528,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,2024-04-26,1,725.4,390470936,92,84274754,...,8.183795,0.015805,0.002629,112.968064,1713126.0,112.968044,1037.475389,2.701114e+05,1.709065e+07,0.000493
1,USUG12400910,Not Like Us,Not Like Us,Kendrick Lamar,2024-05-04,2,545.9,323703884,92,116347040,...,5.914765,0.558451,0.098989,52.206235,3486739.0,52.206158,360.993347,3.827205e+05,6.853257e+05,0.003444
2,QZJ842400387,i like the way you kiss me,I like the way you kiss me,Artemas,2024-03-19,3,538.4,601309283,92,122599116,...,12.244480,0.036389,0.008100,90.948052,2228730.0,90.948022,1204.559279,3.502832e+05,9.626059e+06,0.001514
3,USSM12209777,Flowers,Flowers - Single,Miley Cyrus,2023-01-12,4,444.9,2031280633,85,1096100899,...,13.537537,0.075056,0.009854,150.039823,10629796.0,150.039802,2181.209183,1.401664e+06,1.867484e+07,0.000753
4,USAT22311371,Lovin On Me,Lovin On Me,Jack Harlow,2023-11-10,6,410.1,670665438,83,131148091,...,13.671903,0.044628,0.006479,51.148196,1392593.0,51.148183,750.441209,2.732252e+05,6.122264e+06,0.001471


PCA-Ready Data:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,2.124205,-0.058308,-0.048343,-0.018223,-0.060460,-0.606974,-0.434193,-0.041134,-0.040606,-0.045177,-0.331929,-0.070968,-0.091160,0.163825,7.240441,-0.316038
1,-0.096977,-0.068467,-0.048251,-0.018346,-0.061873,-0.719705,-0.769256,-0.041123,-0.040570,-0.052174,0.032294,-0.084664,-0.109816,0.548603,-0.080167,-0.131227
2,0.928221,-0.055188,-0.048351,-0.018301,-0.060781,-0.565550,0.165439,-0.041133,-0.040604,-0.047713,-0.226046,-0.075931,-0.086552,0.437767,3.909490,-0.252112
3,2.744423,-0.040049,-0.048145,-0.018287,-0.062009,-0.222852,0.356381,-0.041132,-0.040603,-0.040908,1.499169,-0.062611,-0.059619,4.030265,7.947362,-0.299740
4,1.441525,-0.062024,-0.048357,-0.018312,-0.061418,-0.286540,0.376223,-0.041133,-0.040604,-0.052296,-0.397753,-0.084903,-0.099076,0.174465,2.345978,-0.254756


### Plot PCA colored by All Time Rank and Binned All Time Rank

In [24]:
def plot_pca(df_pca, dims, column, title, color_scheme):
    """
    Create 2D and 3D scatterplots of the PCA transformed data points and color by All Time Rank 
    and All Time Rank Bin categories.
    """
    hover_data = ['Track', 'Artist', column]
    
    if dims == 3:
        fig = px.scatter_3d(df_pca, x='PC1', y='PC2', z='PC3', color=column, title=title,
                            color_continuous_scale=color_scheme, hover_data=hover_data)
        fig.update_layout(
            scene_camera=dict(
                eye=dict(x=1.3, y=-1.7, z=1.3),
                center=dict(x=0, y=0, z=0),
                up=dict(x=0, y=0, z=1),
            )
        )
    else:
        fig = px.scatter(df_pca, x='PC1', y='PC2', color=column, title=title,
                         color_continuous_scale=color_scheme, hover_data=hover_data)
    
    fig.update_layout(autosize=False, width=1000, height=800)
    fig.update_traces(marker=dict(size=6, opacity=0.7))
    # save plotly's as json files to display on website
    # fig.write_html(f"./pca-plots/{title.replace(' ', '')}_{dims}D.html")
    fig.show()

def find_cumevr_threshold(df_scaled, threshold):
    """Find number of PCs required to have >=`threshold` variance retention."""
    pca_full = PCA().fit(df_scaled)
    cumevr = np.cumsum(pca_full.explained_variance_ratio_)
    return np.argmax(cumevr >= threshold) + 1, pca_full

def plot_variance(ax1, ax2, evr, cumevr, threshold, pc_crit):
    """Plot EVR and cumulative EVR barplots."""
    
    # barplot of explained variance ratio
    ax1.bar(range(1, len(evr) + 1), evr, color='mediumseagreen', alpha=0.7, width=0.8)
    ax1.set_title('Explained Variance Ratio', fontsize=18)
    ax1.set_ylabel('EVR')
    ax1.set_xlabel('Principal Component')
    ax1.set_xticks(range(1, len(evr) + 1))

    # barplot of cumulative explained variance ratio
    colors = ['black' if i < pc_crit else 'lightgrey' for i in range(len(cumevr))]
    ax2.bar(range(1, len(cumevr) + 1), cumevr, color=colors, width=0.8)
    ax2.axhline(y=threshold, color='mediumseagreen', linestyle=':', label='95% Threshold', lw=3)
    ax2.set_title('Cumulative Explained Variance Ratio', fontsize=18)
    ax2.set_ylabel('Cumulative EVR')
    ax2.set_xlabel('Principal Component')
    ax2.set_xticks(range(1, len(cumevr) + 1))
    ax2.legend()

def plot_pca_heatmap(ax3, pca_full, numerical_cols):
    """Plot PCA loadings, which represent the amount that each variable contributes to a PC."""
    
    # create custom colormap
    colormap_heatmap = mcolors.LinearSegmentedColormap.from_list('custom_colormap', ['black', 'white', 'mediumseagreen'])
    
    # heatmap of the loadings
    sns.heatmap(pd.DataFrame(pca_full.components_.T, index=numerical_cols, 
                             columns=[f'PC{i+1}' for i in range(len(pca_full.components_))]), 
                             cmap=colormap_heatmap, annot=True, fmt='.2f', linewidths=0.5, ax=ax3)
    ax3.set_title('PCA Component Heatmap', fontsize=18)

### Run above functions

In [ ]:
# create custom colormaps
colormap1 = [(0.0, 'mediumseagreen'), (0.15, 'limegreen'), (0.5, 'gold'), (0.75, 'orange'), (1.0, 'red')]
colormap2 = [(0.0, 'black'), (0.5, 'white'), (1.0, 'mediumseagreen')]

"""
Using the PCA function:
"""

# plot data points in the 2D and 3D PC vector spaces
for column, color_scheme in zip(['All Time Rank', 'All Time Rank Bin'], [colormap1, colormap2]):
    plot_pca(pca_df_2d, 2, column, f'PCA Colored by {column}', color_scheme)
    plot_pca(pca_df_3d, 3, column, f'PCA Colored by {column}', color_scheme)

# find number of PCs required to have >=`threshold` variance retention
threshold = 0.95
pc_crit, pca_full = find_cumevr_threshold(df_scaled, threshold)
evr, cumevr = pca_full.explained_variance_ratio_, np.cumsum(pca_full.explained_variance_ratio_)

# plot evr, cumevr, and loadings
fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
plot_variance(axes[0], axes[1], evr, cumevr, threshold, pc_crit)
plot_pca_heatmap(axes[2], pca_full, quantitative_cols)
plt.show()

# print top three largest eigenvalues
top_3_eigenvalues = np.sort(pca_full.explained_variance_)[-3:]
print('Top 3 eigenvalues:')
for eigenvalue in top_3_eigenvalues[::-1]:
    print(f'{eigenvalue:.2f}')
print(f'Sum of all eigenvalues: {np.sum(pca_full.explained_variance_):.2f}')

"""
Using the KernelPCA function with a radial basis function:
"""

for column, color_scheme in zip(['All Time Rank', 'All Time Rank Bin'], [colormap1, colormap2]):
    plot_pca(kpca_df_2d, 2, column, f'RBF Kernel PCA Colored by {column}', color_scheme)
    plot_pca(kpca_df_3d, 3, column, f'RBF Kernel PCA Colored by {column}', color_scheme)
    
# save the three principal component KernelPCA dataframe for clustering
# kpca_df_3d.to_csv('../../data/kpca3d.csv', encoding='utf-8', index=False)